# Equity Cross-Sectional Signal Extraction Case Study

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["PYTHONWARNINGS"] = "ignore"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import statsmodels.api as sm

from IPython.display import display
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def cross_sectional_ic(frame, signal_col, target_col="target_return_next_day", method="spearman"):
    """Daily cross-sectional rank IC (Spearman by default). Returns a Series indexed by date."""
    return frame.groupby("date").apply(
        lambda x: x[[signal_col, target_col]].corr(method=method).iloc[0, 1]
    )


def annualized_sharpe(returns, periods_per_year=252):
    """Annualized Sharpe ratio from a Series of period returns."""
    if returns.std() == 0:
        return np.nan
    return np.sqrt(periods_per_year) * returns.mean() / returns.std()

## 1. Load data and perform a minimal global audit

In [ ]:
path = "equity_cross_section_case.csv"
df_raw = pd.read_csv(path, parse_dates=["date"])

print("Raw shape:", df_raw.shape)
print("Date range:", df_raw["date"].min(), "to", df_raw["date"].max())
print("Assets:", df_raw["asset"].nunique())
print("Duplicate rows:", int(df_raw.duplicated().sum()))
print("Missing values by column:")
display(df_raw.isna().sum().to_frame("missing_count").T)
print("\nCross-sectional coverage by date:")
display(df_raw.groupby("date")["asset"].nunique().describe().to_frame("asset_count").T)

In [ ]:
summary_table = pd.DataFrame({
    "min": df_raw.select_dtypes(include=np.number).min(),
    "max": df_raw.select_dtypes(include=np.number).max(),
    "mean": df_raw.select_dtypes(include=np.number).mean(),
    "std": df_raw.select_dtypes(include=np.number).std(),
})
display(summary_table)

***To be noted***: this is clearly a panel the questions is “can I rank assets cross-sectionally well enough to monetize the dispersion of returns?”

I will perform an intentionally minimal and global cleaning, avoiding any transformation that could use future information:
- sorting by date then asset
- dropping exact duplicates
- winsorizing daily returns at 1% / 99% globally
- keeping missing features for later causal imputation inside the modeling pipeline
- preserving sector information because sector structure will matter both for feature engineering (within-sector relative signals) and for interpreting portfolio results

In [ ]:
df = df_raw.sort_values(["date", "asset"]).drop_duplicates().reset_index(drop=True).copy()
lo, hi = df["return_1d"].quantile([0.01, 0.99])
df["return_1d"] = df["return_1d"].clip(lo, hi)
print("Post-cleaning shape:", df.shape)

## 2. Define the prediction target

In [ ]:
df = df.sort_values(["asset", "date"]).copy()
df["target_return_next_day"] = df.groupby("asset")["return_1d"].shift(-1)
# use only variables observable at date t to predict t+1 return
df = df.dropna(subset=["target_return_next_day"]).reset_index(drop=True).copy()
df[["date", "asset", "return_1d", "target_return_next_day"]].head()

## 3. Reserve a final holdout before detailed EDA

In [ ]:
unique_dates = np.sort(df["date"].unique())
holdout_dates = unique_dates[-int(len(unique_dates)*0.20):]

eda_df = df[~df["date"].isin(holdout_dates)].copy()
holdout_df = df[df["date"].isin(holdout_dates)].copy()

display(pd.DataFrame({
    "sample": ["eda (development)", "final_holdout"],
    "rows": [len(eda_df), len(holdout_df)],
    "start_date": [eda_df["date"].min(), holdout_df["date"].min()],
    "end_date": [eda_df["date"].max(), holdout_df["date"].max()],
    "unique_dates": [eda_df["date"].nunique(), holdout_df["date"].nunique()],
}))

## 4. EDA on the development sample only

### 4.1 Existing features and target

In [ ]:
feature_cols = [
    "value_signal", "quality_signal", "momentum_20d", "short_term_reversal",
    "beta_60d", "liquidity_score", "eps_revision_signal", "size_score",
    "market_return", "market_vol_20d", "macro_regime"
]

eda_corr = eda_df[feature_cols + ["target_return_next_day"]].corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(eda_corr, cmap="coolwarm", center=0)
plt.title("Correlation screen on development sample")
plt.tight_layout()
plt.show()

cs_ic = pd.Series({
    col: cross_sectional_ic(eda_df, col).mean()
    for col in feature_cols
}).sort_values(ascending=False)
display(cs_ic.to_frame("avg_daily_cross_sectional_IC_spearman"))

We already see the broad intended structure: value, quality and earnings-revision type features look directionally helpful on average, while short-term reversal looks negative as expected. Also, market-level variables are mostly common across names on a given date, so I do not expect them to carry large pure cross-sectional power unless they interact with stock-level signals.

In [ ]:
%matplotlib inline

pair_sample = eda_df[["return_1d", "target_return_next_day", "value_signal", "quality_signal", "momentum_20d", "short_term_reversal"]].sample(4000, random_state=0)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
plot_specs = [
    ("value_signal", "target_return_next_day"),
    ("quality_signal", "target_return_next_day"),
    ("momentum_20d", "target_return_next_day"),
    ("short_term_reversal", "target_return_next_day"),
]
for ax, (xcol, ycol) in zip(axes.flatten(), plot_specs):
    sns.scatterplot(data=pair_sample, x=xcol, y=ycol, s=12, alpha=0.25, ax=ax)
    ax.set_title(f"{xcol} vs {ycol}")
plt.tight_layout()
plt.show()

### 4.2 Asset and sector correlation structure

In [ ]:
ret_pivot = eda_df.pivot_table(index="date", columns="asset", values="return_1d")
asset_corr = ret_pivot.corr()
upper_vals = asset_corr.where(np.triu(np.ones(asset_corr.shape), 1).astype(bool)).stack().dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(upper_vals.values, bins=40)
axes[0].set_title("Distribution of pairwise asset correlations")

sector_avg = eda_df.groupby(["date", "sector"])["return_1d"].mean().unstack()
sns.heatmap(sector_avg.corr(), annot=True, cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Sector return correlations")
plt.tight_layout()
plt.show()

upper_vals.describe()

--> Pairwise correlations are meaningfully positive, and sector returns are correlated as well.

### 4.3 Cross-sectional robustness through time

In [ ]:
ic_signals = ["value_signal", "quality_signal", "momentum_20d", "eps_revision_signal"]
daily_ic = pd.DataFrame({
    sig: cross_sectional_ic(eda_df, sig) for sig in ic_signals
})

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for ax, col in zip(axes.flatten(), daily_ic.columns):
    daily_ic[col].plot(ax=ax, alpha=0.8)
    ax.axhline(0, color="black", lw=1)
    ax.set_title(col)
plt.tight_layout()
plt.show()

ic_summary = daily_ic.agg(["mean", "std"]).T
ic_summary["n_days"] = daily_ic.count()
ic_summary["tstat"] = ic_summary["mean"] / (ic_summary["std"] / np.sqrt(ic_summary["n_days"]))
ic_summary["pct_positive"] = (daily_ic > 0).mean()
display(ic_summary)

In [ ]:
# IC summary table above now includes t-stat and pct_positive directly

Value shows a clear and statistically robust signal (t-stat ≈ 5.5 over ~1.5 years of daily data), while quality and momentum exhibit weaker but marginally significant predictive power (t-stats around 2). EPS revisions appear non-informative (t-stat ≈ 0.2). Market-level features (market_return, market_vol_20d, macro_regime) show NaN for cross-sectional IC, which is expected: they take the same value for all assets on a given date, so they carry no cross-sectional variation by construction.

Overall, signals are small relative to noise — typical for daily cross-sectional data — and require aggregation or portfolio construction to be effectively exploited.

### 4.4 Simple economic interpretation check

In [ ]:
eda_df["high_vol"] = (eda_df["market_vol_20d"] > eda_df["market_vol_20d"].median()).astype(int)

rob_rows = []
for reg, sub in eda_df.groupby("high_vol"):
    for sig in ["value_signal", "quality_signal", "momentum_20d", "eps_revision_signal"]:
        ic = cross_sectional_ic(sub, sig).mean()
        rob_rows.append({"high_vol": reg, "signal": sig, "avg_ic": ic})

display(pd.DataFrame(rob_rows).pivot(index="signal", columns="high_vol", values="avg_ic"))

--> A plausible pattern here is that revisions and momentum become more informative in stressed regimes, while slow-moving value remains more stable but weaker day-to-day. I would view that as economically credible rather than suspicious.

## 5. Feature creation for modeling

In [ ]:
df["sector_rel_value"] = df["value_signal"] - df.groupby(["date", "sector"])["value_signal"].transform("mean")
df["sector_rel_quality"] = df["quality_signal"] - df.groupby(["date", "sector"])["quality_signal"].transform("mean")

Important caveat on the existing features: in a real research process I would inspect the exact feature construction logic, especially for momentum/beta/revision variables. Since this is an exploratory notebook, I am implicitly assuming they are point-in-time and causally aligned. That assumption should be audited before trusting any result.

## 6. Development split for model selection

In [ ]:
unique_dates_all = np.sort(df["date"].unique())
holdout_dates_model = unique_dates_all[-int(len(unique_dates_all)*0.20):]
train_dates = unique_dates_all[:int(len(unique_dates_all)*0.60)]
test_dates = unique_dates_all[int(len(unique_dates_all)*0.60):-int(len(unique_dates_all)*0.20)]

train_df = df[df["date"].isin(train_dates)].sort_values(["date", "asset"]).reset_index(drop=True).copy()
test_df = df[df["date"].isin(test_dates)].sort_values(["date", "asset"]).reset_index(drop=True).copy()
holdout_df = df[df["date"].isin(holdout_dates_model)].sort_values(["date", "asset"]).reset_index(drop=True).copy()

# Interaction features: vol threshold computed on train only (no holdout leakage)
vol_median_train = train_df["market_vol_20d"].median()

for split in [train_df, test_df, holdout_df]:
    high_vol = (split["market_vol_20d"] > vol_median_train).astype(int)
    split["epsrev_high_vol"] = split["eps_revision_signal"] * high_vol
    split["mom_high_vol"] = split["momentum_20d"] * high_vol

model_features = [
    "value_signal", "quality_signal", "momentum_20d", "short_term_reversal",
    "beta_60d", "liquidity_score", "eps_revision_signal", "size_score",
    "market_return", "market_vol_20d", "macro_regime", "epsrev_high_vol",
    "mom_high_vol", "sector_rel_value", "sector_rel_quality", "sector"
]

cat_cols = ["sector"]
num_cols = [c for c in model_features if c not in cat_cols]

preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))]), cat_cols),
])

print(f"Train: {len(train_df)} rows, {len(train_dates)} dates")
print(f"Test:  {len(test_df)} rows, {len(test_dates)} dates")
print(f"Holdout: {len(holdout_df)} rows")

## 7. Models

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

X_train = train_df[model_features]
y_train = train_df["target_return_next_day"]
X_test = test_df[model_features]
y_test = test_df["target_return_next_day"]

# TimeSeriesSplit works correctly here because train_df is sorted by (date, asset)
tscv = TimeSeriesSplit(n_splits=4)

ridge_gs = GridSearchCV(
    Pipeline([("prep", preprocess), ("model", Ridge())]),
    param_grid={"model__alpha": [2.0, 8.0, 16.0, 32.0]},
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    n_jobs=-1,
    refit=True,
)

rf_gs = GridSearchCV(
    Pipeline([
        ("prep", preprocess),
        ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
    ]),
    param_grid={
        "model__max_depth": [6, 8],
        "model__min_samples_leaf": [20, 50],
        "model__max_features": ["sqrt", 0.5],
    },
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    n_jobs=-1,
    refit=True,
)

ridge_gs.fit(X_train, y_train)
rf_gs.fit(X_train, y_train)

ridge_pred = ridge_gs.best_estimator_.predict(X_test)
rf_pred = rf_gs.best_estimator_.predict(X_test)

train_mean_target = y_train.mean()
naive_pred = np.repeat(train_mean_target, len(X_test))

res = pd.DataFrame({
    "model": ["best_ridge", "best_random_forest", "historical_mean"],
    "rmse": [
        rmse(y_test, ridge_pred),
        rmse(y_test, rf_pred),
        rmse(y_test, naive_pred),
    ],
    "r2": [
        r2_score(y_test, ridge_pred),
        r2_score(y_test, rf_pred),
        r2_score(y_test, naive_pred),
    ],
})

display(res)
print("Best Ridge params:", ridge_gs.best_params_)
print("Best RF params:", rf_gs.best_params_)

Before over-interpreting the learned model, it is useful to check whether most of the ranking value already comes from a small set of economically sensible signals. 

--> Comparing a single-signal prediction (sector-relative value), the full Ridge, and the random forest helps distinguish signal choice, weight estimation, non-linear structure, and potential dilution from adding broader features.

In [ ]:
# (signal comparison follows directly)

In [ ]:
score_df = test_df[["date", "asset", "sector", "target_return_next_day"]].copy()

# standalone sector-relative value (zero-parameter benchmark)
score_df["sector_rel_value"] = test_df["sector_rel_value"]

# best tuned ridge model
score_df["best_ridge"] = ridge_gs.best_estimator_.predict(test_df[model_features])

# best tuned random forest model
score_df["best_random_forest"] = rf_gs.best_estimator_.predict(test_df[model_features])

models_to_compare = [
    "sector_rel_value",
    "best_ridge",
    "best_random_forest",
]

summary_rows = []
for model_name in models_to_compare:
    ic = cross_sectional_ic(score_df, model_name)

    summary_rows.append({
        "model": model_name,
        "mean_daily_spearman_ic": ic.mean(),
        "std_daily_spearman_ic": ic.std(),
        "tstat_ic": ic.mean() / (ic.std() / np.sqrt(len(ic))) if ic.std() > 0 else np.nan,
        "n_days": len(ic),
    })

summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values("mean_daily_spearman_ic", ascending=False)
    .reset_index(drop=True)
)

display(summary_df)

In [ ]:
rf_feature_names = rf_gs.best_estimator_.named_steps["prep"].get_feature_names_out()
pd.Series(rf_gs.best_estimator_.named_steps["model"].feature_importances_, index=rf_feature_names).sort_values(ascending=False).head(20)

An interesting pattern emerges from the comparison of RMSE and rank IC. Ridge achieves the best RMSE, but the random forest produces the highest cross-sectional Spearman IC. This means Ridge is better at predicting the level of returns, while the random forest is better at ranking assets — and ranking is what matters for a long/short portfolio.

However, the feature importance raises a concern: `macro_regime` alone captures 61% of the random forest's importance, followed by `market_return` (17%) and `market_vol_20d` (10%). These three features are all date-level — they take the same value for every asset on a given date. Purely cross-sectional features (value, quality, momentum, revisions) each contribute less than 2%.

This does not mean the random forest is useless for cross-sectional ranking. A gross Sharpe well above zero in the long/short backtest proves it does generate cross-sectional content. The most likely mechanism is that the forest learns **regime-dependent cross-sectional tilts** — for example, "when macro_regime is high, high-value stocks outperform more." This is economically sensible, but it also means the signal is unstable: every time the regime shifts, the cross-sectional rankings reshuffle, which drives high turnover.

Bottom-line: the random forest remains the strongest ranking model, but its edge over sector-relative value (a zero-parameter signal) is driven by regime interactions that may prove costly to trade. Sector-relative value is the natural benchmark.

**A note on optimization alignment:** the GridSearchCV currently optimizes `neg_root_mean_squared_error`, which rewards accurate level predictions. Since the trading objective is cross-sectional ranking, a custom scorer based on average daily Spearman IC (via `make_scorer`) would be more aligned with the actual goal. This could lead to different hyperparameter choices — in particular, it might favor configurations that sacrifice RMSE for better ranking consistency.

## 9. Translate predictions into a long/short portfolio

In [ ]:
def run_equal_weight_backtest(frame, score_col, target_col="target_return_next_day", q=0.10, cost_bps=5):
    out = []
    prev_w = None

    for d, g in frame.groupby("date"):
        score = g.set_index("asset")[score_col]
        lo = score.quantile(q)
        hi = score.quantile(1 - q)

        long_assets = score.index[score >= hi]
        short_assets = score.index[score <= lo]

        weights = pd.Series(0.0, index=score.index)
        if len(long_assets) > 0:
            weights.loc[long_assets] = 1 / len(long_assets)
        if len(short_assets) > 0:
            weights.loc[short_assets] = -1 / len(short_assets)

        weights = weights / weights.abs().sum()

        if prev_w is None:
            turnover = weights.abs().sum()
        else:
            all_assets = weights.index.union(prev_w.index)
            turnover = (
                weights.reindex(all_assets).fillna(0) - prev_w.reindex(all_assets).fillna(0)
            ).abs().sum()

        asset_returns = g.set_index("asset")[target_col]
        ret = (weights * asset_returns).sum() - (cost_bps / 10000.0) * turnover

        out.append({"date": d, "strategy_return": ret, "turnover": turnover})
        prev_w = weights.copy()

    return pd.DataFrame(out)


strategies = {
    "rf_equal_weight": "best_random_forest",
    "sector_rel_value_equal_weight": "sector_rel_value",
}

results = []
all_bt = {}

for strategy_name, score_col in strategies.items():
    bt = run_equal_weight_backtest(
        score_df,
        score_col=score_col,
        q=0.10,
        cost_bps=5,
    )
    all_bt[strategy_name] = bt

    results.append({
        "strategy": strategy_name,
        "mean_daily_ret": bt["strategy_return"].mean(),
        "daily_vol": bt["strategy_return"].std(),
        "annualized_sharpe": annualized_sharpe(bt["strategy_return"]),
        "avg_turnover": bt["turnover"].mean(),
    })

summary_df = pd.DataFrame(results).sort_values("annualized_sharpe", ascending=False)
display(summary_df)

plt.figure(figsize=(10, 5))
for strategy_name, bt in all_bt.items():
    plt.plot(bt["date"], (1 + bt["strategy_return"]).cumprod(), label=strategy_name)
plt.legend()
plt.title("Development cumulative returns - non sector-neutral")
plt.tight_layout()
plt.show()

In [ ]:
cost_grid = [0, 2, 5, 10, 15, 20]
rows = []

for cost_bps in cost_grid:
    bt = run_equal_weight_backtest(score_df, score_col="best_random_forest", q=0.10, cost_bps=cost_bps)
    rows.append({
        "cost_bps": cost_bps,
        "annualized_sharpe": annualized_sharpe(bt["strategy_return"]),
        "avg_turnover": bt["turnover"].mean(),
    })

pd.DataFrame(rows)

Both strategies generate positive Sharpe ratios on the development test set at 5 bps, with the random forest slightly ahead (0.94 vs 0.84 for sector-relative value). However, the cost sensitivity analysis is sobering: the random forest's gross Sharpe (4.66 at 0 bps) collapses to negative territory by 10 bps. With a daily turnover of 1.52, the signal is almost entirely consumed by realistic transaction costs.

This confirms that the random forest captures genuine cross-sectional content — the gross signal is strong — but the regime-dependent nature of its rankings (driven by macro_regime interactions) makes positions unstable from day to day. The sector-relative value strategy has lower turnover (1.04) and degrades more gracefully with costs, consistent with a slower, more persistent signal.

Note that neither strategy enforces sector neutrality: the backtest simply ranks all assets together and goes long the top decile / short the bottom decile. The sector-relative signal reduces implicit sector bets (because assets are scored relative to their sector peers) but does not eliminate them.

--> Next steps to improve implementation:
- smooth the RF signal over a short rolling window before portfolio construction
- rebalance weekly instead of daily to reduce turnover
- blend today's target portfolio with yesterday's holdings
- test entry/exit buffers so that small score changes do not trigger immediate trades
- evaluate a properly sector-neutral portfolio that selects long/short within each sector

## 10. Final holdout evaluation

In [ ]:
holdout_scores = holdout_df[["date", "asset", "sector", "target_return_next_day"]].copy()
holdout_scores["best_random_forest"] = rf_gs.best_estimator_.predict(holdout_df[model_features])
holdout_scores["sector_rel_value"] = holdout_df["sector_rel_value"]

rows = []
for col in ["best_random_forest", "sector_rel_value"]:
    ic = cross_sectional_ic(holdout_scores, col)
    bt = run_equal_weight_backtest(
        holdout_scores,
        score_col=col,
        q=0.10,
        cost_bps=5,
    )

    rows.append({
        "strategy": col,
        "mean_holdout_ic": ic.mean(),
        "holdout_ic_tstat": ic.mean() / (ic.std() / np.sqrt(len(ic))) if ic.std() > 0 else np.nan,
        "holdout_annualized_sharpe": annualized_sharpe(bt["strategy_return"]),
        "holdout_avg_turnover": bt["turnover"].mean(),
    })

pd.DataFrame(rows)

Both signals retain positive and statistically meaningful holdout IC (random forest: 0.027, t-stat 2.8; sector-relative value: 0.026, t-stat 2.6), so the predictive content does survive out of sample. The two ICs are now nearly identical, whereas the random forest had a clear edge on the development test set (0.052 vs 0.025). This convergence suggests that part of the random forest's development-set advantage came from regime patterns that were less stable out of sample.

Economically, both strategies produce negative Sharpe ratios on holdout after 5 bps costs. The random forest's turnover remains high (~1.49), consuming any gross signal. Sector-relative value is also negative but closer to flat and cheaper to trade.

The takeaway is that the raw signal exists and generalizes, but the current daily-rebalanced equal-weight implementation cannot monetize it after realistic costs.

## 11. Caveat — Sector spread in the data

In [ ]:
# Sector spread analysis — how much of the return dispersion is sector-driven?
sector_returns = df.groupby(["date", "sector"])["target_return_next_day"].mean().unstack()
sector_returns["universe"] = df.groupby("date")["target_return_next_day"].mean()
groups = ["universe"] + sorted(df["sector"].unique())
sector_returns = sector_returns[groups]

sharpe_table = pd.DataFrame(index=groups, columns=groups, dtype=float)
for lg in groups:
    for sg in groups:
        if lg == sg:
            continue
        spread = (sector_returns[lg] - sector_returns[sg]).dropna()
        sharpe_table.loc[lg, sg] = annualized_sharpe(spread)

display(
    sharpe_table.style
    .format("{:.2f}")
    .background_gradient(cmap="coolwarm", axis=None)
    .set_caption("Annualized Sharpe: long sector (rows) vs short sector (columns)")
)

The sector-spread table shows that relative sector bets are meaningful in this sample, with certain long/short sector combinations delivering positive annualized Sharpe ratios. Since the backtest does not enforce sector neutrality, part of the observed portfolio returns may come from implicit sector allocation rather than pure stock selection — even when using a sector-demeaned signal like sector-relative value (which reduces but does not eliminate sector tilts in the top/bottom deciles).

--> A useful next step would be to implement a properly sector-neutral backtest (long/short within each sector) and compare the results to confirm how much of the signal is truly idiosyncratic.

## Final remarks

- **There is meaningful predictive content in the data.** Both the random forest and sector-relative value retain positive holdout IC with t-stats above 2.5. The cross-sectional signal survives out of sample.

- **The random forest captures regime-dependent cross-sectional structure, not pure stock picking.** Feature importance is dominated by date-level variables (macro_regime 61%, market_return 17%, market_vol_20d 10%). The model learns interactions such as "in high-regime environments, certain cross-sectional tilts pay off more" — which is economically sensible but inherently unstable across regime shifts.

- **The gross signal is strong but cannot survive transaction costs.** The random forest achieves a gross Sharpe of 4.66, confirming real cross-sectional content. However, the regime-dependent rankings reshuffle frequently (daily turnover of 1.52), and the Sharpe turns negative by 10 bps of round-trip costs. This is an implementation problem, not a signal problem.

- **Sector-relative value is the more robust benchmark.** It delivers nearly identical holdout IC (0.026 vs 0.027), with lower turnover (1.04 vs 1.49) and no model risk. Within the linear family, adding more features dilutes rather than strengthens the signal — Ridge underperforms the single sector-relative value factor.

- **The backtest is not sector-neutral.** Although sector-relative value reduces implicit sector bets by demeaning within each sector, the portfolio construction ranks all assets together and does not enforce neutrality per sector. Some of the observed returns likely reflect sector allocation.

- **Conclusion and next steps:**
  - The random forest is a promising signal source, but needs a lower-turnover implementation to be monetizable: signal smoothing, weekly rebalancing, or portfolio blending
  - Sector-relative value is the robust baseline — simple, cheap, and stable out of sample
  - A properly sector-neutral backtest (long/short within each sector) is needed to isolate idiosyncratic alpha from sector tilts
  - Replacing the RMSE-based model selection with a rank-IC scorer would better align the optimization objective with the trading objective